# 07 — RAG agéntico (LangChain + LangGraph, LLM vía Ollama)

El mismo agente del repo (`agentic_rag/agent.py`: grafo LangGraph plan → act → observe → answer, 3 herramientas, ReAct textual, máx. 3 pasos) pero con el LLM vía **ChatOllama** contra el servidor local. El repo ya trae ese backend (`agentic_rag/llm_backends.py`, `configure_llm("ollama", ...)`): mismo grafo, mismas herramientas, mismos prompts, mismo scorer; solo cambia el cerebro.

**Modelo por defecto: `qwen2.5:7b`** — el agente es justo el sistema que necesita el modelo más grande: con el 1.5B sacó 2/12 en CPU y nunca usó `vector_search` ni `calculator` (24/24 llamadas fueron `graph_query`). El parámetro `AGENT_OLLAMA_MODEL` permite bajar a `qwen2.5:1.5b` para la comparación en paridad. En la T4 el 7B (Q4_K_M, ~4.7 GB) cabe cómodo.

**Prerrequisitos:** notebooks 01 (chunks) y 02 (grafo Relate cargado + servidor arriba, o al menos el store `femsa.rl` y el binario `relate`).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
AGENT_OLLAMA_MODEL = "qwen2.5:7b"   # agente: 7b (default, el que muestra lo agentico)
                                     # o qwen2.5:1.5b para paridad con el run en CPU (2/12)
TRACE_OUT = REPO + "/agentic_rag/run_trace_agentic_ollama_colab.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 rank-bm25==0.2.2 neo4j>=6.4.1 langchain==1.5.2 langgraph==1.2.24 langchain-core==1.8.5 langchain-huggingface==1.3.0 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=AGENT_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", AGENT_OLLAMA_MODEL)

## Verificar Relate (grafo y binario)

La herramienta `graph_query` usa el CLI `relate` contra el store `pipeline/data/femsa.rl`. Si el notebook 02 no se corrió en esta sesión, el binario y el store pueden faltar: esta celda lo verifica y dice qué correr. (El servidor no hace falta para `graph_query`: el CLI lee el store directo; el servidor solo se necesita para `load_relate.py`.)

In [ ]:
import os, subprocess

RELATE_BIN = os.path.expanduser("~/.local/bin/relate")
DB = REPO + "/pipeline/data/femsa.rl"
print("binario relate:", RELATE_BIN, "->", os.path.exists(RELATE_BIN))
print("store femsa.rl:", DB, "->", os.path.exists(DB))
if not os.path.exists(RELATE_BIN):
    print("Falta el binario: corre el notebook 02 (instala Relate en la instancia).")
if not os.path.exists(DB):
    print("Falta el store: corre el notebook 02 (carga el grafo).")
assert os.path.exists(RELATE_BIN) and os.path.exists(DB), "Prerrequisitos del NB02 incompletos."

## Correr el agente (Ollama 7B)

Importa el agente del repo y lo corre con el backend Ollama. Reanudable: salta las preguntas cuyo id ya está en la traza (cada pregunta tarda minutos; si la sesión se corta, re-corre la celda y sigue donde quedó). La traza es nueva (`run_trace_agentic_ollama_colab.jsonl`) y nunca toca la del 1.5B en CPU.

In [ ]:
import json, sys, time
sys.path.insert(0, REPO + "/agentic_rag")
from agent import build_graph, configure_llm, run_question

configure_llm("ollama", model=AGENT_OLLAMA_MODEL, host="http://localhost:11434")

done = set()
try:
    for line in open(TRACE_OUT):
        try:
            done.add(json.loads(line)["id"])
        except (json.JSONDecodeError, KeyError):
            pass
except FileNotFoundError:
    pass
if done:
    print(f"reanudando: {sorted(done)} ya en {TRACE_OUT}")

qs = json.load(open(REPO + "/memo/eval_questions.json"))["questions"]
graph = build_graph()
t0 = time.time()
with open(TRACE_OUT, "a") as f:
    for q in qs:
        if q["id"] in done:
            print(f"{q['id']} ya hecha, salto", flush=True)
            continue
        print(f"{q['id']} L{q['level']}: {q['question']}", flush=True)
        try:
            rec = run_question(q["question"], q["id"], q["level"], graph)
        except Exception as e:
            import traceback
            traceback.print_exc()
            rec = {"id": q["id"], "level": q["level"], "question": q["question"],
                   "llm_backend": f"ollama {AGENT_OLLAMA_MODEL}",
                   "final_answer": f"ERROR del agente: {type(e).__name__}: {e}",
                   "steps": [], "plan_raw_outputs": [],
                   "seconds": round(time.time() - t0, 1)}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        tools = [s["tool"] for s in rec["steps"]]
        print(f"{q['id']} -> {rec['final_answer'][:160]} "
              f"(tools={tools}, {rec['seconds']}s)", flush=True)
print(f"listo en {time.time() - t0:.1f}s -> {TRACE_OUT}")

## Puntuar con el mismo scorer

Corre `agentic_rag/score_agentic.py` tal cual, con la etiqueta del modelo. En CPU (1.5B) el agente sacó 2/12 (Q4, Q8), 0 alucinaciones: la pregunta es si el 7B usa las herramientas (vector_search, calculator) y sube el marcador.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "score_agentic.py", "--trace", TRACE_OUT,
                    "--out", REPO + "/agentic_rag/results_agentic_ollama_colab.md",
                    "--model-label", f"Ollama {AGENT_OLLAMA_MODEL} (Colab T4)",
                    "--system-name", "RAG agentico Ollama (Colab)"],
                   cwd=REPO + "/agentic_rag")
print("exit:", r.returncode)

## Alternativa documentada (Hugging Face local, no por defecto en Colab)

El mismo agente con el backend local 1.5B en GPU (`configure_llm("local", model="memo/models/Qwen2.5-1.5B-Instruct")`) corre, pero el backend local carga el modelo sin `device_map`, así que en Colab caería en CPU. Por eso el default en Colab es Ollama: el servidor ya gestiona la GPU. Si quieres el 7B por HF con cuantización 4-bit, añade `bitsandbytes` al pip install y adapta `llm_backends.py` con `BitsAndBytesConfig(load_in_4bit=True)` — eso queda como ejercicio documentado, no implementado aquí.